In [1]:
# Parameters — change these and re-run
DATE = "2021-02-21"
PORTFOLIO = "60/40 Multi-Asset"
ESTIMATION_WINDOW = 252
REBALANCE_TOLERANCE = 0.05
CANDIDATE_TRANSFER_WEIGHT = 0.005


In [2]:
import sys
from pathlib import Path

current_directory = Path.cwd().resolve()
project_root = next(
    path for path in (current_directory, *current_directory.parents)
    if (path / "pyproject.toml").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from IPython.display import Markdown, display
from src.display.candidate_trade_impact_renderer import render_candidate_trade_impact_markdown
from src.display.rebalance_review_renderer import render_rebalance_review_markdown
from src.engine.attribution_engine import AttributionEngine
from src.engine.candidate_trade_engine import CandidateTradeEngine
from src.engine.candidate_trade_impact_engine import CandidateTradeImpactEngine
from src.engine.rebalance_trigger_engine import RebalanceTriggerEngine
from src.engine.risk_drift_engine import RiskDriftEngine
from src.infrastructure.in_memory_portfolios import InMemoryPortfolioRepository
from src.infrastructure.yfinance_returns import YFinanceReturnsProvider


In [3]:
# Composition root — generate one candidate and evaluate its hypothetical impact.
portfolios = InMemoryPortfolioRepository()
returns_provider = YFinanceReturnsProvider()
attribution_engine = AttributionEngine(portfolios, returns_provider)
risk_drift_engine = RiskDriftEngine()
rebalance_trigger_engine = RebalanceTriggerEngine()
candidate_trade_engine = CandidateTradeEngine()
candidate_trade_impact_engine = CandidateTradeImpactEngine(
    attribution_engine=attribution_engine,
    risk_drift_engine=risk_drift_engine,
    rebalance_trigger_engine=rebalance_trigger_engine,
)


In [4]:
attribution = attribution_engine.attribute(
    portfolio_name=PORTFOLIO,
    date=DATE,
    estimation_window=ESTIMATION_WINDOW,
)
portfolio = portfolios.get(PORTFOLIO)
risk_drift = risk_drift_engine.calculate(attribution, portfolio)
rebalance_trigger = rebalance_trigger_engine.evaluate(
    risk_drift, tolerance=REBALANCE_TOLERANCE
)
display(Markdown(render_rebalance_review_markdown(
    rebalance_trigger, portfolio_name=PORTFOLIO, date=DATE
)))


**RISK DRIFT — REBALANCE REVIEW**<br><br>**60/40 Multi-Asset**<br>As of 21 February 2021<br><br>

Tolerance band: ±5.00 percentage points<br><br>

**REVIEW REBALANCE**<br><br>

| Asset | Target | Current | Drift | Band | Status | Suggested direction |
|---|---:|---:|---:|---:|---|---|
| IEF | 20% | -3% | -23.35 pp | ±5.00 pp | Outside | Consider increasing |
| SPY | 45% | 67% | +22.27 pp | ±5.00 pp | Outside | Consider reducing |
| EFA | 20% | 31% | +10.65 pp | ±5.00 pp | Outside | Consider reducing |
| GLD | 15% | 5% | -9.57 pp | ±5.00 pp | Outside | Consider increasing |

The trigger identifies risk-contribution drift outside the configured tolerance band. It does not determine trade size, execution, cost, liquidity, or whether the PM should rebalance.



In [5]:
candidate_result = candidate_trade_engine.generate(
    reviewed_drift=rebalance_trigger,
    transfer_weight=CANDIDATE_TRANSFER_WEIGHT,
)
if candidate_result.candidate is None:
    display(Markdown(
        "**CANDIDATE TRADE IMPACT**\n\n"
        "No candidate trade was generated.\n\n"
        f"Reason: {candidate_result.reason}\n\n"
        "Multi-leg, cash-funded, and fallback candidate generation are deferred."
    ))
else:
    candidate = candidate_result.candidate
    display(Markdown(
        f"**Candidate:** Sell {candidate.donor_asset} / Buy {candidate.receiver_asset}\n\n"
        f"Weight transfer: {candidate.transfer_weight:.2%}\n\n"
        "Candidate status: hypothetical"
    ))


**Candidate:** Sell SPY / Buy IEF

Weight transfer: 0.50%

Candidate status: hypothetical

In [6]:
impact = None
if candidate_result.candidate is not None:
    impact = candidate_trade_impact_engine.evaluate(
        portfolio=portfolio,
        candidate=candidate_result.candidate,
        date=DATE,
        estimation_window=ESTIMATION_WINDOW,
        tolerance=REBALANCE_TOLERANCE,
        current_attribution=attribution,
    )


## What the impact measures mean

**Portfolio volatility** measures the typical scale of the portfolio’s return variation over the estimation period. The comparison shows whether the hypothetical trade increased or decreased overall portfolio risk. A lower value does not automatically mean that risk-contribution alignment improved.

**Maximum absolute risk-contribution drift** is the largest distance between an asset’s current risk contribution and its target contribution, ignoring direction. A lower value means the largest individual risk-budget deviation became smaller.

**Total absolute risk-contribution drift** adds the absolute deviations for all assets. It shows how far the whole risk-contribution profile is from its target. A lower value means the combined deviations decreased.

**Assets outside tolerance** counts how many assets are beyond the configured risk-contribution tolerance band. A lower count means fewer assets require review under that rule, but it does not show the size of the remaining deviations.

**Risk-contribution change** is proposed risk contribution minus current risk contribution for each asset. It shows how the hypothetical trade changes each asset’s share of total portfolio risk. All assets can change because of covariance relationships and the portfolio-volatility denominator.

**Drift change** is proposed drift minus current drift for each asset. For an asset with positive drift, a negative change generally moves it toward target. For an asset with negative drift, a positive change generally moves it toward target. The sign must be read together with the original drift direction.

All contribution and drift changes are shown in percentage points. These measures answer different questions; none alone identifies an automatically correct trade.

In [7]:
if impact is not None:
    display(Markdown(render_candidate_trade_impact_markdown(
        impact, portfolio_name=PORTFOLIO, date=DATE
    )))


**CANDIDATE TRADE IMPACT**<br><br>**60/40 Multi-Asset**<br>As of 21 February 2021<br><br>

**Candidate:** Sell SPY / Buy IEF<br>Weight transfer: 0.50 percentage points<br><br>

| Field | Value |
|---|---|
| Donor | SPY |
| Receiver | IEF |
| Donor weight change | -0.50 pp |
| Receiver weight change | +0.50 pp |
| Selection rule | Largest positive drift to largest negative drift |
| Candidate status | Hypothetical |

This is a hypothetical equal-and-opposite weight transfer. It is not an optimised rebalance or an executable order.

**Portfolio-level impact**

| Measure | Current | Proposed | Change |
|---|---:|---:|---:|
| Portfolio volatility | 0.0122 | 0.0121 | -0.0001 |
| Maximum absolute RC drift | 23.35 pp | 23.42 pp | +0.07 pp |
| Total absolute RC drift | 65.84 pp | 65.82 pp | -0.02 pp |
| Assets outside tolerance | 4 | 4 | +0 |

**Asset-level impact**

| Asset | Current RC | Proposed RC | RC change | Current drift | Proposed drift | Drift change |
|---|---:|---:|---:|---:|---:|---:|
| IEF | -3.35% | -3.42% | -0.07 pp | -23.35 pp | -23.42 pp | -0.07 pp |
| SPY | 67.27% | 66.98% | -0.28 pp | +22.27 pp | +21.98 pp | -0.28 pp |
| EFA | 30.65% | 30.93% | +0.27 pp | +10.65 pp | +10.93 pp | +0.27 pp |
| GLD | 5.43% | 5.51% | +0.08 pp | -9.57 pp | -9.49 pp | +0.08 pp |

The candidate is shown for hypothetical impact assessment.


This notebook tests one equal-and-opposite fixed-weight transfer. The current and proposed calculations use the same observation date, return data, estimation window, covariance methodology, mandate, and tolerance rule. The candidate is hypothetical: this slice does not optimise, execute, price, or automatically recommend a rebalance.